# Wireless Networking and Mobile Computing

### 1 IEEE 802.11 Reference Model

IEEE 802.11 separates functions into the physical layer and the medium access control layer. The physical layer handles modulation, coding, and transmission over the radio channel. The MAC layer handles channel access, framing, acknowledgments, association, and link security. In addition, the MAC Layer Management Entity coordinates management operations such as scanning and authentication. The Station Management Entity spans both layers and provides a control and configuration interface toward higher layers through Service Access Points.

**Layer roles**

- Layer 1: Physical transmission schemes. Modulation, coding, timing, synchronization, channel estimation, OFDM symbol generation

- Layer 2: MAC protocols and MAC Layer Management Entity. Channel access, CSMA CA, interframe spacing, retransmissions, association and authentication

- Layer 1 and 2: Station Management Entity. Configuration, scanning, timing alignment, power management, measurements, key management hooks

- Higher layers: Interfaced through Service Access Points (SAPs)

<center>
  <img src="../../Admin/wifi/wifi_osi.png"
       alt="IEEE 802.11 Reference Model"
       width="30%">
</center>

#### 1.1 MAC Entities and Architecture

A Wi-Fi station implements both PHY and MAC, with the MAC responsible for medium access, frame exchange, retransmission, and coordination. Access points form a Basic Service Set (BSS) and distribute frames through the network. Independent BSSs form ad-hoc networks without infrastructure. The MAC Layer Management Entity (MLME) manages association, scanning, authentication, power saving, and channel switching. Station Management Entity functions span PHY and MAC, while Service Access Points define the interfaces toward higher protocol layers. This reference layer architecture lets the MAC evolve while maintaining a consistent service model.

<center>
  <img src="../../Admin/wifi/wifi_architecture.png"
       alt="IEEE 802.11 Architecture"
       width="30%">
</center>

#### 1.2 Frame Exchange and Reliability

Wi-Fi uses positive acknowledgments for reliability. A sender transmits a data frame and expects an ACK after a short fixed time. If no ACK arrives, the frame is retransmitted following the contention rules. Short data units may be fragmented into multiple MAC Protocol Data Units to limit collision impact, and each fragment is acknowledged individually. Receivers reassemble fragments into complete frames. This acknowledgment-driven reliability model is fundamental to 802.11 and ensures dependable delivery over an unreliable wireless medium.

<center>
  <img src="../../Admin/wifi/wifi_exchange.png"
       alt="IEEE 802.11 Example Frame Exchange"
       width="40%">
</center>


### 2 Distributed Coordination Function (DCF)

DCF is the fundamental access mechanism in IEEE 802.11. It uses carrier sensing, inter-frame timing, and random backoff to coordinate transmissions in a distributed wireless environment without collision detection.


* Physical carrier sensing
  Detects energy or preambles to determine whether the channel is occupied

* Virtual carrier sensing (NAV)
  Tracks medium reservations announced in control frames to avoid interfering with ongoing exchanges

* Inter-frame spacing (SIFS, DIFS, optional PIFS)
  Guarantees priority for immediate responses such as ACKs before new contention begins

* Random backoff and contention window
  A station waits a random number of slots before transmission, and doubles the contention window after each failed attempt to reduce future collisions

* Optional RTS/CTS handshake
  Exchanges short Request-to-Send and Clear-to-Send frames before sending data to mitigate hidden-node collisions and reserve the medium for large frames



<center>
  <img src="../../Admin/wifi/wifi_dcf.png"
       alt="IEEE 802.11 Frame Spacing"
       width="35%">
</center>

#### 2.1 Carrier Sensing and Channel Access

DCF follows a listen-before-talk principle. A station first performs Clear Channel Assessment (CCA) to check whether the channel is idle, based on energy detection or a detected preamble. If the channel is free, it waits a short fixed interval and then transmits. If the medium is busy, the station defers and enters contention. Virtual carrier sensing uses the Network Allocation Vector to track the expected duration of other transmissions, preventing overlap. Together, physical and virtual sensing form the core of collision avoidance in Wi-Fi.

<center>
  <img src="../../Admin/wifi/wifi_carrierSensing.png"
       alt="IEEE 802.11 Carrier Sensing"
       width="35%">
</center>

#### 2.2 Inter-Frame Spacing


Inter-frame spaces determine access priority and ensure orderly medium use.

* **SIFS (Short Inter-Frame Space)**
  Used for immediate responses such as ACKs and CTS.
  Highest priority. Allows fast turnaround between receive and transmit.

* **DIFS (Distributed Inter-Frame Space)**
  Used before a station begins contention for a new transmission.
  Ensures ACKs and control responses happen first.

* **(Optional) PIFS (PCF Inter-Frame Space)**
  Used by the Point Coordination Function, if enabled, to gain priority between SIFS and DIFS.

These timing gaps allow acknowledgments and control frames to preempt new transmissions and give the radio hardware time to switch roles when required.



#### 2.3 Contention and Backoff

If a station cannot transmit immediately, it selects a random backoff value from the current contention window. The counter decreases only while the channel remains idle and freezes during activity. After each failed attempt the contention window doubles up to a maximum, reducing collision probability under heavy load. When the counter reaches zero, the station transmits. This exponential backoff procedure allows DCF to scale in congested environments while maintaining fairness among contending stations.

<center>
  <img src="../../Admin/wifi/wifi_backoff.png"
       alt="IEEE 802.11 Backoff Window"
       width="35%">
</center>

#### 2.4 RTS / CTS 

RTS/CTS provides an optional handshake before sending a data frame. A station first sends a short Request-to-Send, the receiver answers with Clear-to-Send, and only then the data frame is transmitted. Other stations that overhear either control frame set their NAV and hold off, so the medium is reserved for the duration of the exchange. This reduces the chance of collisions for long frames, but because it adds overhead, it is typically used selectively based on frame size or channel conditions.

Hidden stations occur when two transmitters are out of each other’s range but can both reach the same receiver. Since they cannot sense one another’s transmissions, carrier sense alone cannot prevent collisions. RTS/CTS addresses this by letting the receiver broadcast the CTS, which both transmitters can hear. When a hidden station hears the CTS, it updates its NAV and stays silent, preventing overlapping transmissions at the receiver. This targeted reservation makes RTS/CTS especially useful in dense or obstructed environments where hidden nodes are likely.

<center>
  <img src="../../Admin/wifi/wifi_rts.png"
       alt="IEEE 802.11 RTS / CTS"
       width="45%">
</center>

### 3. Reliability Mechanisms
Reliability mechanisms strengthen basic contention access by reducing the impact of interference and hidden nodes, ensuring frames can be delivered even in busy or challenging radio environments.

#### 3.1 Fragmentation and Defragmentation

Fragmentation splits a large MAC service data unit into multiple smaller MAC protocol data units to improve robustness. Smaller fragments limit the cost of collisions, because only the failed portion needs to be retransmitted. Each fragment is individually acknowledged and protected by the NAV, ensuring ordered delivery. The receiver reassembles all fragments into the original frame once they are successfully received. This mechanism is most useful in noisy or congested environments, though it adds overhead and is less common in modern deployments where aggregation dominates.

<center>
  <img src="../../Admin/wifi/wifi_fragmentation.png"
       alt="IEEE 802.11 Fragmentation Example"
       width="35%">
</center>

### 3. Performance and Service Enhancements

These mechanisms build on basic contention access to improve throughput, prioritize traffic, and reduce overhead at high PHY rates.


#### 3.1 Enhanced Distributed Channel Access (EDCA)

EDCA extends the basic contention mechanism to support Quality-of-Service. Instead of treating all traffic equally, frames are assigned to four access categories: background, best-effort, video, and voice. Each category competes for the medium independently, allowing latency-sensitive streams to access the channel sooner.


| Access Category     | CWmin | CWmax | AIFSN | AIFS  |
| ------------------- | ----- | ----- | ----- | ----- |
| Legacy (DCF)        | 15    | 1023  | 2     | 34 µs |
| AC_BK (Background)  | 15    | 1023  | 7     | 79 µs |
| AC_BE (Best Effort) | 15    | 1023  | 3     | 43 µs |
| AC_VI (Video)       | 7     | 15    | 2     | 34 µs |
| AC_VO (Voice)       | 3     | 7     | 2     | 34 µs |


Inside a station, each access category runs its own backoff engine. If two categories inside the same device finish backoff at the same time, a virtual collision resolution ensures that the higher-priority queue transmits while the lower-priority one retries. This internal arbitration prevents a single device’s queues from colliding with each other and improves responsiveness for time-critical flows.

<center>
  <img src="../../Admin/wifi/ecda_qos.png"
       alt="IEEE 802.11 ECDA Quality of Service"
       width="40%">
</center>


Priority is controlled by changing contention parameters. Higher-priority traffic uses shorter inter-frame timing (AIFS) and smaller contention windows (CWmin and CWmax). For example, voice traffic has the shortest wait and smallest contention windows, followed by video, while best-effort and background traffic wait longer. This creates a probability-based prioritization rather than strict scheduling and keeps the system compatible with legacy devices.

<center>
  <img src="../../Admin/wifi/ecda_throughput_3.png"
       alt="IEEE 802.11 ECDA Throughput"
       width="40%">
  <img src="../../Admin/wifi/ecda_throughput_16.png"
    alt="IEEE 802.11 ECDA Throughput (up to 16)"
    width="40%">
</center>

### 4 Centralized Coordination and Advanced MAC Features

#### 4.1 MAC Superframe: Contention-Free and Contention Periods

802.11 divides time into optional coordination phases. During the contention-free period, a hybrid coordinator schedules transmission rights so stations can send without competing for the medium. During the contention period, stations use EDCA with carrier sensing and backoff. Most commercial networks simply operate in contention mode, but the superframe structure provides a mechanism for deterministic access when required.

<center>
  <img src="../../Admin/wifi/wifi_superframe.png"
       alt="802.11 Periodic Superframe Period"
       width="40%">
</center>

#### 4.2 HCCA (Hybrid Coordination Function Controlled Access)

HCCA is the centralized counterpart to EDCA. An access point acting as a hybrid coordinator assigns stations transmission opportunities directly, bypassing contention and enabling strict latency and jitter control. It is intended for real-time applications and environments requiring guaranteed service, but it is rarely used in practice. EDCA remains the de-facto QoS method while HCCA completes the standard’s QoS model.

#### 4.3 TXOP (Transmission Opportunity)

A TXOP gives a station a time-bounded window to send multiple frames once it gains channel access. Rather than contending for each frame, the station transmits a burst separated only by short inter-frame spacing. TXOP limits differ by access category, allowing voice and video streams to transmit efficiently with minimal overhead while still bounding channel occupation time.

#### 4.4 MAC Support for MIMO Operation

Modern Wi-Fi PHY layers (starting with 802.11n) introduced multiple-antenna techniques such as spatial multiplexing, diversity, and beamforming. Although MIMO processing happens in the physical layer, the MAC plays a key coordination role. It advertises MIMO capabilities, negotiates the number of spatial streams, and supports aggregation to keep the medium efficiently used at high PHY rates. Combined with rate control, the MAC adapts modulation, coding, and spatial stream count based on link conditions. The result is higher throughput, improved reliability in fading channels, and more efficient spectrum use without changing the distributed access model.

<center>
  <img src="../../Admin/wifi/wifi_mimo.png"
       alt="802.11 Multiple Input Multiple Output"
       width="50%">
</center>

#### 4.5 Block Acknowledgment


Block acknowledgment increases MAC efficiency by confirming multiple frames in one control exchange rather than acknowledging each frame individually. A station requests a block ACK once or implicitly signals it through aggregated transmission, and the receiver replies with a bitmap indicating which subframes were received. This enables selective retransmission instead of repeating the whole burst. Later refinements reduce overhead further, including compressed block ACK formats with smaller bitmaps and multi-TID block ACKs that support parallel traffic streams. 

Depending on traffic requirements, block ACK can operate in immediate mode, where feedback follows right after reception, or delayed mode, where acknowledgment arrives later to suit buffered or scheduled access. Together with aggregation, block ACK is essential for sustaining modern Wi-Fi PHY rates by minimizing control overhead and retransmission cost.


<center>
  <img src="../../Admin/wifi/wifi_blockACK.png"
       alt="802.11 Block Acknowledgement"
       width="40%">
</center>

#### 4.6 Basic Service Set (BSS) Architecture

A Basic Service Set defines the stations that share a coordinated MAC domain, typically under one access point. When a BSS operates in isolation, contention and delay remain predictable because one coordinator controls timing and channel access scheduling. 

In dense deployments, however, multiple BSSs can operate on the same channel and interfere. Overlapping access points do not coordinate, and their hybrid coordinators or EDCA queues may attempt transmission at the same time. This leads to collisions, unpredictable delay, and degraded Quality of Service, especially for time-sensitive traffic. Although mechanisms such as TXOP limits and channel planning reduce impact, fully reliable QoS across overlapping BSSs remains an unresolved coexistence challenge.

<center>
  <img src="../../Admin/wifi/bss_overlap.png"
       alt="BSS Overlap in Sending"
       width="40%">

  <img src="../../Admin/wifi/bss_performance.png"
       alt="BSS Overlap Performance Bottleneck"
       width="40%">
</center>
